<a href="https://colab.research.google.com/github/luiscano2330-bot/curso-programacion-cientifica/blob/main/Entregable_2_visualizacion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Proyecto integrador — Grupo 2: Incidentes de seguridad en Medellín (2025)

**Entregable 1 — Análisis y limpieza de datos con pandas**

- **Integrantes:** LUIS STEVEN CANO
- **Grupo:** 2
- **Repositorio:** `proyecto-integrador-pc-grupo-2`



## 1. Carga e inspección del conjunto de datos

Antes de analizar nada necesito saber qué tengo: cuántas filas y columnas hay, qué tipo tiene cada columna y qué valores aparecen. El archivo se lee con una **ruta relativa** (`data/...`) porque así el notebook funciona igual en mi computador y en Colab cuando el CSV está en la carpeta `data/` del repositorio.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 120)

ruta_csv = "data/seguridad_incidentes_medellin_2025.csv"

# En Colab el CSV se sube al panel de archivos; si quedó en la raíz, lo paso a data/
if not Path(ruta_csv).exists():
    Path("data").mkdir(exist_ok=True)
    if Path("seguridad_incidentes_medellin_2025.csv").exists():
        Path("seguridad_incidentes_medellin_2025.csv").replace(ruta_csv)
    else:
        from google.colab import files
        print("Sube el archivo seguridad_incidentes_medellin_2025.csv")
        subidos = files.upload()
        Path(next(iter(subidos))).replace(ruta_csv)

df = pd.read_csv(ruta_csv)
print("El archivo tiene", df.shape[0], "filas y", df.shape[1], "columnas")

Sube el archivo seguridad_incidentes_medellin_2025.csv


El archivo se cargó completo: **4.505 filas y 19 columnas**. Cada fila es un incidente reportado en Medellín durante 2025 y cada columna es una característica de ese incidente (fecha, comuna, tipo, arma, víctima, etc.). No hubo errores de lectura, así que el separador y la codificación del archivo son los esperados.

In [ ]:
df.head(5)

In [ ]:
df.info()

In [ ]:
df.dtypes.to_frame("tipo_de_dato")

`head()` muestra que las columnas de texto traen tildes y espacios tal como las escribió quien reportó el incidente, y `info()` confirma que **todas las columnas tienen 4.505 datos, salvo seis que tienen menos**: `iluminacion_zona`, `edad_victima`, `bien_hurtado`, `valor_perdido_cop`, `tiempo_respuesta_min` y `testigos`. Esas ausencias son el primer problema de calidad que voy a revisar.

En los tipos aparece algo raro: **`fecha` está guardada como texto (`str`)** y no como fecha. Mientras sea texto no puedo ordenar por fecha ni sacar el mes con `.dt`, así que la corrijo más abajo. `hora` y `mes` sí son enteros, y las demás columnas numéricas son decimales.

In [ ]:
df.describe().round(2)

`describe()` resume las columnas numéricas. Lo primero que salta a la vista es `valor_perdido_cop`: la media (≈ 2,79 millones) es casi el doble de la mediana (≈ 1,45 millones) y el máximo llega a **120 millones de pesos**. Esa diferencia entre media y mediana avisa de valores extremos que voy a revisar.

`edad_victima` va de 12 a 72 años, un rango creíble. `hora` va de 0 a 23 y `mes` de 1 a 12, así que no hay horas ni meses imposibles. `tiempo_respuesta_min` va de 3 a 122 minutos y `testigos` de 0 a 6.

In [ ]:
print("Tipos de incidente:")
print(df["tipo_incidente"].value_counts())
print()
print("Lugares:")
print(df["lugar"].value_counts())
print()
print("Armas:")
print(df["arma"].value_counts())
print()
print("Comunas distintas escritas:", df["comuna"].nunique())
print(df["comuna"].value_counts().head(20))

Los **hurtos** son el grupo más grande: hurto a persona (1.367) y hurto de celular (1.252) suman más de la mitad de los casos, y a ellos se suma el hurto de moto (401). Los delitos de convivencia (riña, violencia intrafamiliar y lesiones) son menos frecuentes.

La mayoría de los incidentes ocurre en **vía pública** (2.457 de 4.505), lo que tiene sentido para hurtos y riñas.

En `comuna` aparece el primer problema serio: **52 formas distintas de escribir solo 16 comunas**. "La Candelaria", "LA CANDELARIA", "la candelaria" y " La Candelaria" son la misma comuna, pero pandas las cuenta como grupos separados. Si agrupo sin corregir, los promedios de esas comunas salen con muy pocos datos y no sirven.

In [ ]:
print("Antes:", df["fecha"].dtype)
df["fecha"] = pd.to_datetime(df["fecha"])
print("Después:", df["fecha"].dtype)
print("Primer día:", df["fecha"].min().date(), "| Último día:", df["fecha"].max().date())
print("Días distintos con reportes:", df["fecha"].nunique())

Convierto `fecha` a tipo fecha porque el diccionario dice que es una fecha y porque así puedo usar `.dt` para sacar el mes o el día de la semana. La conversión funcionó sin errores (no hubo fechas mal escritas) y cubre todo 2025: del **1 de enero al 31 de diciembre**, con 365 días distintos. Ese es el periodo que voy a analizar.

## 2. Diagnóstico de calidad de los datos

Ya sé qué hay. Ahora reviso **qué está mal**: faltantes, faltantes que en realidad significan "no aplica", valores imposibles, códigos raros, categorías inconsistentes y filas repetidas. La idea es tener la evidencia antes de decidir qué hacer con cada problema.

In [ ]:
tabla_faltantes = pd.DataFrame({
    "faltantes": df.isna().sum(),
    "porcentaje": (df.isna().mean() * 100).round(1),
})
tabla_faltantes.sort_values("faltantes", ascending=False).head(10)

Seis columnas tienen faltantes:

- `testigos`: **3.224 de 4.505 (71,6 %)**. Es demasiado: con casi tres de cada cuatro datos vacíos no se puede concluir nada serio.
- `bien_hurtado` y `valor_perdido_cop`: **1.485 cada una (33,0 %)**. El mismo número en las dos columnas sugiere que faltan juntas, y eso hay que verificarlo.
- `tiempo_respuesta_min`: 550 (12,2 %).
- `edad_victima`: 393 (8,7 %).
- `iluminacion_zona`: 219 (4,9 %).

No todos esos vacíos significan lo mismo, así que los reviso por separado antes de rellenar nada.

In [ ]:
pd.crosstab(df["tipo_incidente"], df["bien_hurtado"].isna(), margins=True)

Aquí está la clave: `bien_hurtado` y `valor_perdido_cop` **solo faltan en riñas, lesiones personales y violencia intrafamiliar**, y no falta ni una en los hurtos. Es decir, no son datos perdidos: son **"no aplica"**. En una riña no hay un bien hurtado ni un valor perdido que registrar.

Eso cambia la decisión: **estos vacíos no se imputan**. Rellenarlos con la mediana inventaría un robo que nunca ocurrió y además mezclaría delitos de hurto con delitos de convivencia.

In [ ]:
def contar_atipicos(serie):
    """Cuenta los valores fuera del rango Q1 - 1.5*IQR a Q3 + 1.5*IQR."""
    q1 = serie.quantile(0.25)
    q3 = serie.quantile(0.75)
    iqr = q3 - q1
    limite_inferior = q1 - 1.5 * iqr
    limite_superior = q3 + 1.5 * iqr
    fuera = (serie < limite_inferior) | (serie > limite_superior)
    return int(fuera.sum()), round(limite_inferior, 2), round(limite_superior, 2)

for columna in ["edad_victima", "valor_perdido_cop", "tiempo_respuesta_min"]:
    cantidad, limite_inferior, limite_superior = contar_atipicos(df[columna])
    print(f"{columna}: {cantidad} atípicos | rango normal [{limite_inferior} ; {limite_superior}]")

In [ ]:
df.loc[df["valor_perdido_cop"] > 50_000_000,
       ["id_incidente", "comuna", "tipo_incidente", "bien_hurtado", "valor_perdido_cop"]
      ].sort_values("valor_perdido_cop", ascending=False)

El criterio IQR marca 13 edades, 331 valores perdidos y 153 tiempos de respuesta. Pero **"atípico" no es lo mismo que "error"**, así que miro cada caso:

- **Edad (13 casos):** son personas de más de 61 años. Un adulto mayor víctima de un delito es perfectamente posible, así que se conservan.
- **Tiempo de respuesta (153 casos):** son demoras de más de 56 minutos. Son demoras reales, no errores de digitación. Se conservan.
- **Valor perdido (331 casos):** la mayoría son motos y computadores caros, que son plausibles. Pero al filtrar los valores por encima de 50 millones aparecen **5 casos imposibles**: cuatro registros de **120.000.000 COP** (dos celulares y dos bolsos) y una moto de 52.589.000 COP. Un celular no vale 120 millones; eso es un error de digitación (probablemente ceros de más). Esos 5 sí se corrigen.

Esto es importante para la exposición: el criterio automático solo **señala**; la decisión de borrar o conservar la toma quien conoce el dato.

In [ ]:
print("Formas de escribir las comunas:", df["comuna"].nunique())
print(df["comuna"].value_counts().tail(12))
print()
print("Valores de la columna arma:")
print(df["arma"].value_counts())

Además de las comunas repetidas con mayúsculas, minúsculas y espacios, aparece un código raro: **`arma = "Sin dato"` en 314 casos (7,0 %)**. No es un vacío técnico, es una categoría que significa "no se logró identificar el arma". La dejo como categoría propia y no la imputo, porque cambiarla por "Sin arma" afirmaría algo que no sé.

No hay otras categorías mal escritas en `tipo_incidente`, `modalidad`, `lugar` ni `sexo_victima`.

In [ ]:
print("Filas completamente repetidas:", df.duplicated().sum())
print("Identificadores repetidos:", df["id_incidente"].duplicated().sum())
df.loc[df.duplicated(keep=False), ["id_incidente", "fecha", "comuna", "tipo_incidente"]].sort_values("id_incidente").head(8)

Hay **25 filas completamente repetidas** (el mismo incidente con el mismo `id_incidente` y todos los mismos valores). No son dos incidentes distintos: son el mismo registro duplicado, probablemente por una doble carga del archivo. Si no las quito, los conteos por comuna y por tipo quedan inflados y los promedios se calculan dos veces sobre el mismo caso.

## 3. Limpieza de los datos

Con el diagnóstico hecho, aplico las decisiones una por una y **anoto el costo de cada una**: cuántas filas o valores cambian. Trabajo sobre una copia (`df_limpio`) para no dañar los datos originales y poder comparar.

El plan es:

1. Eliminar las 25 filas duplicadas.
2. Unificar las 52 formas de escribir las comunas.
3. Eliminar la columna `testigos` (71,6 % de faltantes).
4. Corregir los 5 valores imposibles de `valor_perdido_cop`.
5. Imputar `iluminacion_zona` con la **moda**.
6. Imputar `edad_victima` con la **mediana global**.
7. Imputar `tiempo_respuesta_min` con la **mediana por comuna**.
8. No tocar los faltantes que significan "no aplica".

In [ ]:
filas_antes = len(df)

df_limpio = df.drop_duplicates().reset_index(drop=True)

print("Filas antes:", filas_antes)
print("Filas después:", len(df_limpio))
print("Filas eliminadas:", filas_antes - len(df_limpio))
print("Identificadores repetidos que quedan:", df_limpio["id_incidente"].duplicated().sum())

**Costo:** se pierden **25 filas** (0,6 % del total) y cada `id_incidente` queda una sola vez. Es un costo bajo y evita contar dos veces los mismos incidentes.

In [ ]:
import unicodedata

def normalizar_texto(texto):
    """Pasa un texto a minúsculas, sin espacios externos y sin tildes."""
    limpio = str(texto).strip().lower()
    sin_tildes = "".join(
        caracter for caracter in unicodedata.normalize("NFKD", limpio)
        if not unicodedata.combining(caracter)
    )
    return sin_tildes

mapa_comunas = {
    "la candelaria": "La Candelaria", "el poblado": "El Poblado",
    "laureles estadio": "Laureles Estadio", "belen": "Belén",
    "robledo": "Robledo", "buenos aires": "Buenos Aires",
    "la america": "La América", "aranjuez": "Aranjuez",
    "castilla": "Castilla", "manrique": "Manrique",
    "san javier": "San Javier", "popular": "Popular",
    "guayabal": "Guayabal", "doce de octubre": "Doce de Octubre",
    "villa hermosa": "Villa Hermosa", "santa cruz": "Santa Cruz",
}

df_limpio["comuna_original"] = df_limpio["comuna"]
df_limpio["comuna"] = df_limpio["comuna"].map(normalizar_texto).map(mapa_comunas)
comunas_corregidas = int((df_limpio["comuna_original"] != df_limpio["comuna"]).sum())
df_limpio = df_limpio.drop(columns="comuna_original")

print("Comunas distintas antes: 52")
print("Comunas distintas después:", df_limpio["comuna"].nunique())
print("Filas cuya comuna se corrigió:", comunas_corregidas)
print("Comunas sin nombre reconocido:", df_limpio["comuna"].isna().sum())
print()
print(df_limpio["comuna"].value_counts())

**Costo:** se reescriben **77 filas** (las que estaban en mayúsculas, minúsculas o con espacios), pero **no se elimina ninguna**. Las 52 categorías quedan en **16 comunas** y ninguna fila se queda sin comuna. Ahora sí puedo agrupar por comuna sin que un promedio salga calculado con 1 o 2 casos mal escritos.

In [ ]:
columnas_antes = df_limpio.shape[1]
faltantes_testigos = int(df_limpio["testigos"].isna().sum())
porcentaje_testigos = df_limpio["testigos"].isna().mean() * 100

df_limpio = df_limpio.drop(columns="testigos")

print("Columnas antes:", columnas_antes, "| Columnas después:", df_limpio.shape[1])
print(f"testigos tenía {faltantes_testigos} faltantes ({porcentaje_testigos:.1f} %)")
print("Filas: no cambian ->", len(df_limpio))

**Decisión:** elimino `testigos` porque **7 de cada 10 celdas están vacías** (3.204 de 4.480, el 71,5 %). Imputarla con la moda (que es 0) diría que en casi todos los incidentes no hubo testigos, cuando en realidad no sabemos si los hubo: es un dato faltante, no un cero. Conservar la columna con ese nivel de vacío solo aportaría ruido.

**Costo:** se pierde una columna, pero **no se pierde ninguna fila**; las otras 18 columnas siguen completas.

In [ ]:
imposibles = df_limpio["valor_perdido_cop"] > 50_000_000
print("Valores imposibles encontrados:", int(imposibles.sum()))

df_limpio.loc[imposibles, "valor_perdido_cop"] = np.nan

# Se rellena con la mediana del mismo tipo de bien, no con la mediana general
df_limpio["valor_perdido_cop"] = df_limpio["valor_perdido_cop"].fillna(
    df_limpio.groupby("bien_hurtado")["valor_perdido_cop"].transform("median")
)

print("Media del valor perdido:", round(df_limpio["valor_perdido_cop"].mean(), 0))
print("Mediana del valor perdido:", round(df_limpio["valor_perdido_cop"].median(), 0))
print("Máximo después de la corrección:", round(df_limpio["valor_perdido_cop"].max(), 0))

**Decisión:** convierto los 5 valores imposibles en faltantes y los reemplazo por la **mediana del mismo tipo de bien** (la mediana de los celulares para un celular, la de los bolsos para un bolso, etc.). Uso la mediana del grupo y no la general porque una moto y un celular no valen lo mismo: la mediana general le asignaría a un celular un valor de moto.

**Costo:** se corrigen **5 valores** y no se pierde ninguna fila. El efecto se nota en la media, que baja de ≈ 2,79 a ≈ 2,62 millones, mientras la mediana casi no se mueve (1,45 millones). Eso confirma que los 120 millones estaban arrastrando el promedio.

In [ ]:
faltantes_iluminacion = int(df_limpio["iluminacion_zona"].isna().sum())
moda_iluminacion = df_limpio["iluminacion_zona"].mode()[0]

df_limpio["iluminacion_zona"] = df_limpio["iluminacion_zona"].fillna(moda_iluminacion)

print("Moda de iluminacion_zona:", moda_iluminacion)
print("Valores imputados:", faltantes_iluminacion)
print("Faltantes que quedan:", int(df_limpio["iluminacion_zona"].isna().sum()))
print()
print(df_limpio["iluminacion_zona"].value_counts(normalize=True).mul(100).round(1))

**Decisión:** `iluminacion_zona` es una variable **categórica** con solo tres valores (Buena, Regular, Deficiente) y un 4,9 % de faltantes. Para una categórica la estrategia adecuada es la **moda**, que es "Buena" (53,4 % de los casos). No uso la media porque no tiene sentido promediar palabras, y no uso la mediana por grupo porque el porcentaje perdido es bajo y no cambia las conclusiones.

**Costo:** se imputan **219 valores** (4,9 % de la columna) y no se pierde ninguna fila.

In [ ]:
print("Mediana de edad por tipo de incidente:")
print(df_limpio.groupby("tipo_incidente")["edad_victima"].agg(["count", "median"]))
print()
print("Mediana de edad por sexo:")
print(df_limpio.groupby("sexo_victima")["edad_victima"].agg(["count", "median"]))

faltantes_edad = int(df_limpio["edad_victima"].isna().sum())
mediana_edad = df_limpio["edad_victima"].median()
df_limpio["edad_victima"] = df_limpio["edad_victima"].fillna(mediana_edad)

print()
print("Mediana global usada:", mediana_edad, "| Valores imputados:", faltantes_edad)

**Decisión:** uso la **mediana global** (31 años) para `edad_victima`. Antes comprobé que la mediana es la misma (30 o 31 años) en todos los tipos de incidente y en los dos sexos, así que no hay un grupo al que le corresponda un valor distinto: imputar por grupo daría prácticamente el mismo resultado. Uso la mediana y no la media porque la mediana no se deja arrastrar por las edades extremas.

**Costo:** se imputan **392 valores** (8,8 %) y no se pierde ninguna fila.

In [ ]:
print("Mediana del tiempo de respuesta por comuna (minutos):")
print(df_limpio.groupby("comuna")["tiempo_respuesta_min"].agg(["count", "median"]).sort_values("median"))

faltantes_tiempo = int(df_limpio["tiempo_respuesta_min"].isna().sum())
df_limpio["tiempo_respuesta_min"] = df_limpio["tiempo_respuesta_min"].fillna(
    df_limpio.groupby("comuna")["tiempo_respuesta_min"].transform("median")
)

print()
print("Valores imputados:", faltantes_tiempo)
print("Faltantes que quedan:", int(df_limpio["tiempo_respuesta_min"].isna().sum()))

**Decisión:** aquí sí uso la **mediana por comuna**. La tabla muestra que el tiempo de respuesta no es igual en toda la ciudad: en Doce de Octubre la mediana es 34 minutos y en Buenos Aires es 17. Si imputara con una mediana global (20 minutos) le pondría a una comuna lenta un tiempo que no le corresponde y borraría justamente la diferencia que quiero analizar.

**Costo:** se imputan **548 valores** (12,2 %) y no se pierde ninguna fila.

In [ ]:
resumen_limpieza = pd.DataFrame([
    {"decision": "Eliminar filas duplicadas", "filas_antes": 4505, "filas_despues": 4480, "valores_afectados": 25},
    {"decision": "Unificar comunas (52 -> 16)", "filas_antes": 4480, "filas_despues": 4480, "valores_afectados": comunas_corregidas},
    {"decision": "Eliminar columna testigos", "filas_antes": 4480, "filas_despues": 4480, "valores_afectados": faltantes_testigos},
    {"decision": "Corregir valores imposibles", "filas_antes": 4480, "filas_despues": 4480, "valores_afectados": 5},
    {"decision": "Imputar iluminacion_zona (moda)", "filas_antes": 4480, "filas_despues": 4480, "valores_afectados": faltantes_iluminacion},
    {"decision": "Imputar edad_victima (mediana global)", "filas_antes": 4480, "filas_despues": 4480, "valores_afectados": faltantes_edad},
    {"decision": "Imputar tiempo_respuesta_min (mediana por comuna)", "filas_antes": 4480, "filas_despues": 4480, "valores_afectados": faltantes_tiempo},
])
resumen_limpieza

In [ ]:
print("Forma final:", df_limpio.shape)
print()
print("Faltantes que quedan por columna:")
print(df_limpio.isna().sum()[df_limpio.isna().sum() > 0])
print()
print("Filas sin bien_hurtado:", int(df_limpio["bien_hurtado"].isna().sum()))
print("Filas sin valor_perdido_cop:", int(df_limpio["valor_perdido_cop"].isna().sum()))

Al final quedan **4.480 filas y 18 columnas**, y los únicos faltantes que sobreviven son los de `bien_hurtado` y `valor_perdido_cop` (1.477 cada uno). Esos son exactamente los **"no aplica"** de riñas, lesiones y violencia intrafamiliar, y se dejan así a propósito: un vacío que significa "no hubo bien hurtado" es información, no un error.

Las columnas que sí usé en el análisis (`comuna`, `iluminacion_zona`, `edad_victima`, `tiempo_respuesta_min`) quedaron completas.

## 4. Preguntas de análisis

Con los datos limpios respondo cinco preguntas que sirven para tomar decisiones de seguridad en la ciudad. Todas usan **filtrado booleano**, **`loc`**, **`sort_values`** y **`groupby` con agregaciones**, e incluyo `count` para saber con cuántos datos se calculó cada promedio.

### Pregunta 1. ¿Qué tipo de incidente es el más frecuente y cuánto pesa en el total?

Sirve para saber dónde se concentra el esfuerzo de prevención. Uso `value_counts()` y ordeno con `sort_values()`.

In [ ]:
conteo_tipos = df_limpio["tipo_incidente"].value_counts().to_frame("incidentes")
conteo_tipos["porcentaje"] = (conteo_tipos["incidentes"] / len(df_limpio) * 100).round(1)
conteo_tipos.sort_values("incidentes", ascending=False)

**Respuesta:** el hurto a persona es el más frecuente con **1.359 casos (30,3 %)**, seguido del hurto de celular con **1.245 (27,8 %)**. Si se suman las tres modalidades de hurto (persona, celular y moto) se llega a **3.003 casos, el 67,0 % del total**. Los delitos de convivencia (riña, violencia intrafamiliar y lesiones) son el 33,0 % restante.

**Implicación:** la prevención del hurto, y en particular del hurto a persona y de celular, es la prioridad si el objetivo es bajar el número total de incidentes.

### Pregunta 2. ¿En qué comunas se concentran los hurtos y cuánto vale lo que se pierde?

Uso `loc` para quedarme solo con los hurtos, filtro booleano para los casos de mayor valor y `groupby` con `count` y `median`.

In [ ]:
hurtos = df_limpio.loc[df_limpio["tipo_incidente"].str.startswith("Hurto"), :]
print("Hurtos en la base:", len(hurtos))

hurtos_por_comuna = hurtos.groupby("comuna").agg(
    hurtos=("id_incidente", "count"),
    valor_mediano=("valor_perdido_cop", "median"),
)
hurtos_por_comuna["porcentaje"] = (hurtos_por_comuna["hurtos"] / len(hurtos) * 100).round(1)
hurtos_por_comuna.sort_values("hurtos", ascending=False).round(0)

In [ ]:
hurtos.loc[hurtos["valor_perdido_cop"] > 15_000_000,
           ["id_incidente", "comuna", "bien_hurtado", "valor_perdido_cop"]
          ].sort_values("valor_perdido_cop", ascending=False).head(8)

**Respuesta:** los hurtos se concentran en tres comunas: **La Candelaria (611, 20,0 %)**, **El Poblado (384, 13,0 %)** y **Laureles Estadio (329, 11,0 %)**. Entre las tres reúnen **1.324 hurtos, el 44,1 %** de todos los hurtos de la ciudad. En las comunas con menos hurtos, como Santa Cruz (68) o Popular (84), el valor mediano perdido es parecido al de las comunas grandes (alrededor de 1,2 a 1,6 millones), así que la diferencia está en la **cantidad** de casos, no en el monto.

Los casos de mayor valor son casi todos **motocicletas** (entre 20 y 32 millones), y La Candelaria aparece varias veces en esa lista.

**Implicación:** una campaña concentrada en tres comunas cubriría casi la mitad de los hurtos. Para las motos, que son lo más costoso, conviene reforzar parqueaderos y controles en La Candelaria.

### Pregunta 3. ¿Qué comunas tienen el tiempo de respuesta más lento?

El tiempo de respuesta es lo que más se puede mejorar con logística. Agrupo por comuna y comparo la mediana, que no se deja arrastrar por las demoras extremas.

In [ ]:
tiempo_por_comuna = df_limpio.groupby("comuna").agg(
    incidentes=("id_incidente", "count"),
    tiempo_mediano=("tiempo_respuesta_min", "median"),
)
tiempo_por_comuna.sort_values("tiempo_mediano", ascending=False).round(1)

**Respuesta:** la mediana del tiempo de respuesta va de **17 minutos en Buenos Aires a 34 minutos en Doce de Octubre**, es decir, el doble. Las comunas más lentas son **Doce de Octubre (34), Santa Cruz (33), Popular (32,5) y Villa Hermosa (32)**; las más rápidas son Buenos Aires (17), La Candelaria, Aranjuez y Robledo (18).

Las comunas lentas no son las que tienen más incidentes (La Candelaria, la que más tiene, responde en 18 minutos). Eso sugiere que el problema no es el volumen de casos sino la **ubicación o la capacidad de respuesta** en la periferia.

**Implicación:** reubicar o reforzar unidades en el nororiente y el occidente (Doce de Octubre, Santa Cruz, Popular, Villa Hermosa) reduciría la brecha de respuesta.

### Pregunta 4. ¿Qué proporción de los incidentes se denuncia, según el tipo?

La denuncia es la puerta de entrada a la investigación. Comparo el porcentaje con `groupby`, `count` y `sum` sobre una columna booleana.

In [ ]:
df_limpio["denuncio"] = df_limpio["denunciado"] == "Sí"

denuncias_por_tipo = df_limpio.groupby("tipo_incidente").agg(
    incidentes=("id_incidente", "count"),
    denuncias=("denuncio", "sum"),
)
denuncias_por_tipo["porcentaje_denuncia"] = (
    denuncias_por_tipo["denuncias"] / denuncias_por_tipo["incidentes"] * 100
).round(1)
denuncias_por_tipo.sort_values("porcentaje_denuncia", ascending=False)

**Respuesta:** en total se denuncia el **48,1 %** de los incidentes. El porcentaje más alto es el del **hurto de moto (65,2 % de 399 casos)** y el más bajo el de la **riña (36,6 % de 609 casos)**. La violencia intrafamiliar (52,7 %) y el hurto de celular (51,0 %) quedan por encima del promedio; el hurto a persona (45,6 %) y las lesiones personales (42,4 %) por debajo.

Tiene sentido que la moto se denuncie más: es un bien caro y muchas veces asegurado, así que la denuncia sirve para el seguro. La riña, en cambio, se resuelve en el momento y se denuncia menos.

**Implicación:** las cifras de riñas y lesiones están más subestimadas que las de hurto. Si se quiere medir la violencia interpersonal, hay que acompañar la denuncia con campañas o con registros de urgencias.

### Pregunta 5. ¿Cómo se relacionan la iluminación de la zona y el tipo de incidente?

Es una pregunta de prevención situacional: si los delitos de convivencia ocurren más en zonas con mala iluminación, la intervención es distinta a la del hurto. Uso una tabla cruzada con porcentajes por fila.

In [ ]:
iluminacion_por_tipo = pd.crosstab(
    df_limpio["tipo_incidente"],
    df_limpio["iluminacion_zona"],
    normalize="index",
).mul(100).round(1)
iluminacion_por_tipo.sort_values("Deficiente", ascending=False)

In [ ]:
df_limpio.groupby("iluminacion_zona").agg(
    incidentes=("id_incidente", "count"),
    camaras_promedio=("camaras_cercanas", "mean"),
    tiempo_mediano=("tiempo_respuesta_min", "median"),
).round(2)

**Respuesta:** los delitos de convivencia ocurren en zonas con **peor iluminación** que los hurtos. El **32,0 % de las violencias intrafamiliares** y el **31,0 % de las riñas** están en zonas de iluminación deficiente, frente al **13,9 % del hurto a persona**. Los hurtos, en cambio, se concentran en zonas de iluminación buena (58,9 % del hurto a persona).

La segunda tabla muestra que la iluminación viene acompañada de otros factores: las zonas de iluminación **deficiente tienen en promedio 1,91 cámaras cercanas y una respuesta mediana de 26 minutos**, mientras las de iluminación **buena tienen 2,75 cámaras y 19 minutos**.

**Implicación:** la iluminación no es la única causa, pero las zonas deficientes acumulan tres desventajas a la vez (menos luz, menos cámaras y respuesta más lenta). Ahí es donde una intervención integral tendría más efecto.

## 5. Conclusiones parciales

1. **La base es utilizable después de la limpieza.** De 4.505 filas quedaron **4.480** (solo se eliminaron 25 duplicados, 0,6 %), con 18 columnas. Las columnas usadas en el análisis quedaron completas.

2. **El problema principal es el hurto.** El 67,0 % de los incidentes son hurtos, y el hurto a persona y el de celular son el 58,1 % del total.

3. **La concentración geográfica es fuerte.** La Candelaria, El Poblado y Laureles Estadio reúnen el 44,1 % de los hurtos.

4. **La respuesta no es igual en toda la ciudad.** La mediana va de 17 a 34 minutos según la comuna, y las comunas lentas no son las de más casos.

5. **La denuncia depende del tipo de delito.** El hurto de moto se denuncia en el 65,2 % de los casos y la riña en el 36,6 %, así que las cifras de violencia interpersonal están más subestimadas.

6. **Iluminación, cámaras y tiempo de respuesta van juntos.** Las zonas con iluminación deficiente tienen menos cámaras (1,91 frente a 2,75) y respuesta más lenta (26 frente a 19 minutos).

**Qué queda por explorar con las gráficas (Entregable 2):** cómo se distribuyen las edades, cómo cambia el número de incidentes mes a mes, qué tan relacionadas están las variables numéricas y cuánto pesa cada tipo de bien hurtado. Eso es lo que voy a mostrar con Matplotlib y Seaborn en el siguiente entregable.

## 6. Visualizaciones con Matplotlib y Seaborn

Trabajo sobre `df_limpio`, los datos ya limpios del Entregable 1. Primero tres gráficas **univariadas** (una variable) y después varias **multivariadas** (dos o más variables). Cada gráfica se guarda en `visualizaciones/` en PNG con un nombre que describe su contenido, y debajo de cada una explico qué pregunta responde, qué se observa y qué implica.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.dpi"] = 110

Path("visualizaciones").mkdir(exist_ok=True)
print("Carpeta de visualizaciones lista")

### 6.1 Univariadas

**Gráfica 1 — Histograma de la edad de las víctimas (Matplotlib).** Es una variable numérica continua, así que el histograma es la gráfica adecuada para ver la forma de la distribución.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.hist(df_limpio["edad_victima"], bins=20, color="#4c72b0", edgecolor="white")
ax.axvline(df_limpio["edad_victima"].median(), color="#c44e52", linestyle="--", linewidth=2,
           label=f"Mediana = {df_limpio['edad_victima'].median():.0f} años")
ax.set_title("Distribución de la edad de las víctimas")
ax.set_xlabel("Edad de la víctima (años)")
ax.set_ylabel("Número de incidentes")
ax.legend()
plt.tight_layout()
plt.savefig("visualizaciones/distribucion-edad-victimas.png", dpi=150, bbox_inches="tight")
plt.show()

**Qué pregunta responde:** ¿las víctimas son principalmente jóvenes o la edad está repartida?

**Qué se observa:** la distribución tiene un solo pico alrededor de los **31 años** (la mediana y la media son 31), con la mayoría de los casos entre **24 y 39 años** (el 50 % central). Hay pocos casos por encima de 60 y el máximo es 72 años; el mínimo es 12.

**Qué implica:** las acciones de prevención deberían hablarle a población adulta joven, no solo a adolescentes. La forma es bastante simétrica, así que la mediana representa bien al grupo.

**Gráfica 2 — Incidentes por tipo (Matplotlib).** Para una variable categórica lo correcto es un gráfico de barras con el conteo de cada categoría. Lo hago horizontal para que los nombres largos se lean sin superponerse.

In [ ]:
conteo_tipos = df_limpio["tipo_incidente"].value_counts().sort_values()

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.barh(conteo_tipos.index, conteo_tipos.values, color="#55a868")
for posicion, valor in enumerate(conteo_tipos.values):
    ax.text(valor + 15, posicion, f"{valor} ({valor / len(df_limpio) * 100:.1f} %)",
            va="center", fontsize=9)
ax.set_title("Número de incidentes por tipo (Medellín, 2025)")
ax.set_xlabel("Número de incidentes")
ax.set_ylabel("Tipo de incidente")
ax.set_xlim(0, conteo_tipos.max() * 1.18)
plt.tight_layout()
plt.savefig("visualizaciones/incidentes-por-tipo.png", dpi=150, bbox_inches="tight")
plt.show()

**Qué pregunta responde:** ¿cuál es el peso relativo de cada tipo de incidente?

**Qué se observa:** el hurto a persona (**1.359; 30,3 %**) y el hurto de celular (**1.245; 27,8 %**) están muy por encima del resto. En el otro extremo, las lesiones personales (**377; 8,4 %**) y el hurto de moto (**399; 8,9 %**) son los menos frecuentes.

**Qué implica:** la diferencia entre el primero y el último es de casi **4 veces**. Cualquier plan que reparta los recursos por igual entre tipos estaría mal priorizado: el hurto a persona y el de celular merecen la mayor parte.

**Gráfica 3 — Diagrama de caja del tiempo de respuesta (Seaborn).** La caja resume una variable numérica: mediana, cuartiles y valores extremos. Sirve para ver de un vistazo el valor típico y la dispersión.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.6))
sns.boxplot(x=df_limpio["tiempo_respuesta_min"], color="#4c72b0", width=0.45, ax=ax)
ax.set_title("Tiempo de respuesta de las autoridades")
ax.set_xlabel("Tiempo de respuesta (minutos)")
ax.set_ylabel("")
plt.tight_layout()
plt.savefig("visualizaciones/tiempo-respuesta-caja.png", dpi=150, bbox_inches="tight")
plt.show()

**Qué pregunta responde:** ¿cuál es el tiempo de respuesta típico y qué tan disperso es?

**Qué se observa:** la mediana es **20 minutos** y la mitad central de los casos está entre **14 y 31 minutos**. La media (23,9) es mayor que la mediana porque hay una cola de casos lentos que llega hasta **122 minutos**.

**Qué implica:** el tiempo típico es razonable, pero la cola de casos de más de 56 minutos (153 casos) es la que eleva el promedio. Mejorar esos casos extremos tiene más efecto sobre la media que bajar un minuto a todos.

**Gráfica 4 — Incidentes por comuna, top 10 (Seaborn).** Otra categórica, ahora con muchas categorías; muestro solo las 10 primeras para que las etiquetas se lean.

In [ ]:
top_comunas = df_limpio["comuna"].value_counts().head(10)

fig, ax = plt.subplots(figsize=(9, 4.8))
sns.barplot(x=top_comunas.values, y=top_comunas.index, hue=top_comunas.index,
            palette="viridis", legend=False, ax=ax)
ax.set_title("Las 10 comunas con más incidentes")
ax.set_xlabel("Número de incidentes")
ax.set_ylabel("Comuna")
plt.tight_layout()
plt.savefig("visualizaciones/incidentes-por-comuna.png", dpi=150, bbox_inches="tight")
plt.show()

**Qué pregunta responde:** ¿en qué comunas se concentra el total de incidentes?

**Qué se observa:** **La Candelaria** encabeza con **746 incidentes (16,7 % del total)**, seguida de **El Poblado (457)** y **Laureles Estadio (383)**. La décima, Manrique, tiene **226**. La diferencia entre la primera y la décima es de más de 3 veces.

**Qué implica:** La Candelaria sola concentra casi uno de cada seis incidentes de la ciudad. Es la primera candidata para cualquier piloto de intervención.

### 6.2 Multivariadas

**Gráfica 5 — Dispersión de edad contra valor perdido (Matplotlib).** Dos variables numéricas; el color separa por sexo, así que la gráfica relaciona tres variables. Solo incluyo hurtos, porque en los demás delitos no hay bien hurtado.

In [ ]:
hurtos = df_limpio.loc[df_limpio["tipo_incidente"].str.startswith("Hurto"), :]

fig, ax = plt.subplots(figsize=(9, 5))
for sexo, color in [("F", "#c44e52"), ("M", "#4c72b0")]:
    datos = hurtos[hurtos["sexo_victima"] == sexo]
    ax.scatter(datos["edad_victima"], datos["valor_perdido_cop"] / 1_000_000,
               s=14, alpha=0.45, color=color, label=f"Sexo {sexo} (n = {len(datos)})")
ax.set_title("Edad de la víctima y valor de lo hurtado")
ax.set_xlabel("Edad de la víctima (años)")
ax.set_ylabel("Valor de lo hurtado (millones de COP)")
ax.set_ylim(0, 35)
ax.legend(title="Sexo de la víctima")
plt.tight_layout()
plt.savefig("visualizaciones/edad-vs-valor-perdido.png", dpi=150, bbox_inches="tight")
plt.show()

**Qué pregunta responde:** ¿las víctimas de mayor edad pierden bienes más caros?

**Qué se observa:** la nube es **horizontal**, sin inclinación. La correlación entre edad y valor perdido es prácticamente cero (**-0,00**), y el valor mediano se mantiene cerca de **1,45 millones de COP** en todas las edades. Los puntos altos (motos de 20 a 32 millones) aparecen en edades muy distintas. Los dos sexos se mezclan sin un patrón claro.

**Qué implica:** la edad no sirve para predecir cuánto va a perder una víctima, así que las medidas de protección no deberían segmentarse por edad. Lo que explica el monto es el **tipo de bien** (una moto vale mucho más que un celular), no quién lo lleva.

**Gráfica 6 — Tiempo de respuesta por tipo de incidente (Seaborn).** Una numérica contra una categórica; la caja permite comparar las distribuciones completas, no solo el promedio.

In [ ]:
orden_tipos = (df_limpio.groupby("tipo_incidente")["tiempo_respuesta_min"]
               .median().sort_values().index)

fig, ax = plt.subplots(figsize=(9, 5))
sns.boxplot(data=df_limpio, x="tiempo_respuesta_min", y="tipo_incidente",
            order=orden_tipos, hue="tipo_incidente", palette="Set2",
            legend=False, ax=ax)
ax.set_title("Tiempo de respuesta según el tipo de incidente")
ax.set_xlabel("Tiempo de respuesta (minutos)")
ax.set_ylabel("Tipo de incidente")
plt.tight_layout()
plt.savefig("visualizaciones/tiempo-respuesta-por-tipo.png", dpi=150, bbox_inches="tight")
plt.show()

**Qué pregunta responde:** ¿los delitos de convivencia tardan más en atenderse que los hurtos?

**Qué se observa:** sí. Las medianas más bajas son **hurto de celular (19 minutos)** y **hurto a persona (20)**; las más altas son **violencia intrafamiliar (26)**, **lesiones personales (25)** y **riña (24)**. La diferencia es de unos **7 minutos**, y las cajas de los delitos de convivencia son más anchas hacia la derecha.

**Qué implica:** los casos de violencia interpersonal, que son los que pueden escalar a algo peor, son justamente los que más esperan. Ahí hay una oportunidad clara de mejora en el despacho.

**Gráfica 7 — Evolución mensual: hurtos y otros delitos (Matplotlib).** Con una fecha se puede ver la evolución en el tiempo. Uso dos líneas porque son dos series y así la leyenda tiene sentido.

In [ ]:
df_limpio["es_hurto"] = df_limpio["tipo_incidente"].str.startswith("Hurto")
mensual = df_limpio.pivot_table(index="mes", columns="es_hurto",
                                values="id_incidente", aggfunc="count")
nombres_meses = ["Ene", "Feb", "Mar", "Abr", "May", "Jun",
                 "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"]

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(mensual.index, mensual[True], marker="o", linewidth=2,
        color="#c44e52", label="Hurtos")
ax.plot(mensual.index, mensual[False], marker="s", linewidth=2,
        color="#4c72b0", label="Otros delitos")
ax.set_title("Incidentes por mes en Medellín (2025)")
ax.set_xlabel("Mes del año")
ax.set_ylabel("Número de incidentes")
ax.set_xticks(range(1, 13))
ax.set_xticklabels(nombres_meses)
ax.legend(title="Grupo de delito")
plt.tight_layout()
plt.savefig("visualizaciones/evolucion-mensual-hurtos.png", dpi=150, bbox_inches="tight")
plt.show()

**Qué pregunta responde:** ¿hay meses con más incidentes y el patrón es igual para hurtos y para otros delitos?

**Qué se observa:** las dos series se mueven casi en paralelo y el pico está en **diciembre**, con **332 hurtos y 177 otros delitos** (**509 incidentes en total**, frente a un promedio mensual de 373). El mes más tranquilo es **febrero (328)**. El resto del año se mantiene entre 340 y 418.

**Qué implica:** diciembre necesita un refuerzo estacional, sobre todo para hurtos. Como las dos series suben juntas, el operativo de fin de año conviene que cubra también riñas y violencia, no solo hurto.

**Gráfica 8 — Mapa de calor de correlaciones (Seaborn).** Un mapa de calor resume muchas relaciones numéricas en una sola imagen; los valores cercanos a 0 indican que no hay relación lineal.

In [ ]:
columnas_numericas = ["hora", "mes", "camaras_cercanas", "edad_victima",
                      "valor_perdido_cop", "tiempo_respuesta_min"]
correlaciones = df_limpio[columnas_numericas].corr()

fig, ax = plt.subplots(figsize=(7.5, 6))
sns.heatmap(correlaciones, annot=True, fmt=".2f", cmap="coolwarm",
            vmin=-1, vmax=1, square=True, linewidths=0.5, ax=ax)
ax.set_title("Correlación entre las variables numéricas")
plt.tight_layout()
plt.savefig("visualizaciones/correlacion-variables-numericas.png", dpi=150, bbox_inches="tight")
plt.show()

**Qué pregunta responde:** ¿qué variables numéricas se mueven juntas?

**Qué se observa:** casi todas las correlaciones son **cercanas a 0**. La única que se destaca es **cámaras cercanas y tiempo de respuesta: -0,23**, negativa y de magnitud moderada: donde hay más cámaras, el tiempo de respuesta tiende a ser menor. Edad, hora y valor perdido no se relacionan entre sí.

**Qué implica:** la ubicación (cámaras) pesa más que las características de la víctima o del bien. Es evidencia débil (correlación no es causa), pero apunta a que la infraestructura de la zona sí importa.

**Gráfica 9 — Valor mediano perdido por tipo de bien (Matplotlib).** Es una numérica resumida por categoría; en vez de cajas uso barras con la mediana porque el objetivo es comparar montos típicos.

In [ ]:
medianas_bien = (hurtos.groupby("bien_hurtado")["valor_perdido_cop"]
                 .median().sort_values() / 1_000_000)

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.barh(medianas_bien.index, medianas_bien.values, color="#8172b2")
for posicion, valor in enumerate(medianas_bien.values):
    ax.text(valor + 0.12, posicion, f"{valor:.2f} M", va="center", fontsize=9)
ax.set_title("Valor mediano de lo hurtado según el tipo de bien")
ax.set_xlabel("Valor mediano (millones de COP)")
ax.set_ylabel("Bien hurtado")
ax.set_xlim(0, medianas_bien.max() * 1.15)
plt.tight_layout()
plt.savefig("visualizaciones/valor-mediano-por-bien.png", dpi=150, bbox_inches="tight")
plt.show()

**Qué pregunta responde:** ¿qué bienes hurtados representan la mayor pérdida económica?

**Qué se observa:** la **motocicleta** es la más costosa con una mediana de **8,07 millones de COP**, muy por encima del **computador (3,46 M)**, la **bicicleta (1,94 M)** y el **celular (1,60 M)**. Al final quedan las **joyas (1,41 M)**, el **bolso (0,71 M)** y el **dinero (0,28 M)**.

**Qué implica:** la moto es **5 veces** más costosa que el celular, aunque se hurta 3 veces menos (399 frente a 1.245). Si el criterio es la pérdida económica y no el número de casos, el hurto de moto merece más atención de la que sugiere su frecuencia.

**Gráfica 10 — Tiempo de respuesta según la iluminación (Seaborn).** Relaciona una numérica con una categórica ordenada (Buena, Regular, Deficiente) y completa la historia de la pregunta 5: iluminación, cámaras y respuesta.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
sns.boxplot(data=df_limpio, x="iluminacion_zona", y="tiempo_respuesta_min",
            order=["Buena", "Regular", "Deficiente"], hue="iluminacion_zona",
            palette="crest", legend=False, ax=ax)
ax.set_title("Tiempo de respuesta según la iluminación de la zona")
ax.set_xlabel("Iluminación de la zona")
ax.set_ylabel("Tiempo de respuesta (minutos)")
plt.tight_layout()
plt.savefig("visualizaciones/tiempo-respuesta-por-iluminacion.png", dpi=150, bbox_inches="tight")
plt.show()

**Qué pregunta responde:** ¿las zonas peor iluminadas también tienen peor respuesta?

**Qué se observa:** la mediana sube de **19 minutos en zonas con iluminación buena** a **26 minutos en zonas con iluminación deficiente** (19 en las regulares). La caja de las zonas deficientes también es más alta y más ancha. A esto se suma lo de la pregunta 5: las zonas deficientes tienen **1,91 cámaras** en promedio frente a **2,75** en las buenas.

**Qué implica:** hay tres factores que se acumulan en las mismas zonas: menos luz, menos cámaras y respuesta más lenta. Una intervención integral ahí (alumbrado, cámaras y unidades) atacaría las tres cosas a la vez.

## 7. Conclusiones y recomendaciones

**Respuestas a las preguntas del proyecto**

1. **Tipo más frecuente:** hurto a persona (1.359; 30,3 %) y hurto de celular (1.245; 27,8 %). Los hurtos son el 67,0 % de los incidentes.
2. **Comunas con más hurtos:** La Candelaria (611), El Poblado (384) y Laureles Estadio (329); entre las tres, el 44,1 % de los hurtos.
3. **Tiempo de respuesta:** va de 17 minutos (Buenos Aires) a 34 (Doce de Octubre); las comunas lentas no son las de más casos.
4. **Denuncia:** 48,1 % en total; máximo en hurto de moto (65,2 %) y mínimo en riña (36,6 %).
5. **Iluminación y tipo:** los delitos de convivencia ocurren más en zonas de iluminación deficiente (32,0 % de la violencia intrafamiliar) que los hurtos (13,9 % del hurto a persona).

**Recomendaciones**

1. **Concentrar la prevención del hurto en tres comunas.** La Candelaria, El Poblado y Laureles Estadio reúnen el 44,1 % de los hurtos (1.324 de 3.003). Un piloto allí cubre casi la mitad del problema con menos recursos.
2. **Reforzar el despacho en el nororiente y el occidente.** Doce de Octubre (34 min), Santa Cruz (33), Popular (32,5) y Villa Hermosa (32) duplican el tiempo de Buenos Aires (17). Reubicar unidades o acortar el trayecto reduciría esa brecha.
3. **Tratar la moto como prioridad económica, no solo por frecuencia.** Su valor mediano es 8,07 millones de COP, cinco veces el de un celular. Parqueaderos vigilados y controles en La Candelaria, donde aparecen los casos más costosos, atacan la mayor pérdida económica.
4. **Intervención integral en zonas de iluminación deficiente.** Allí coinciden menos cámaras (1,91 frente a 2,75) y respuesta más lenta (26 frente a 19 minutos). Alumbrado, cámaras y despacho en el mismo plan atacan los tres factores a la vez.
5. **Reforzar diciembre.** Es el mes con más incidentes (509, frente a un promedio de 373), y suben tanto hurtos como otros delitos.

## 8. Limitaciones

- **Los datos son sintéticos.** Se generaron para el curso a partir de fenómenos reales de Colombia: los lugares existen, las cifras no. Las conclusiones valen para este dataset y **no se pueden generalizar a Medellín ni a la realidad**.
- **No hay población en riesgo.** El dataset cuenta incidentes reportados, no todos los que ocurrieron. Sin la población de cada comuna no se puede calcular una tasa por habitante, así que una comuna con más casos no es necesariamente más peligrosa: puede tener más gente o más reportes.
- **Subregistro distinto por tipo.** La denuncia varía del 36,6 % (riña) al 65,2 % (hurto de moto). Las comparaciones entre tipos están afectadas por ese subregistro desigual.
- **Datos imputados.** Se imputaron 219 valores de `iluminacion_zona`, 392 de `edad_victima` y 548 de `tiempo_respuesta_min`. Son porcentajes bajos o medios, pero introducen supuestos; la imputación por comuna suaviza las diferencias dentro de cada comuna.
- **Se eliminó `testigos`** por tener 71,5 % de faltantes, así que no se puede analizar el efecto de los testigos en la denuncia.
- **Correlación no implica causa.** La relación de -0,23 entre cámaras y tiempo de respuesta, y la de iluminación con el tipo de delito, son asociaciones. No demuestran que poner cámaras reduzca el tiempo de respuesta.
- **Algunos grupos son pequeños.** Santa Cruz tiene 152 incidentes y algunas categorías de bien hurtado tienen menos de 200 casos, así que sus medianas son menos estables.

---
**Fin del proyecto.** El notebook se ejecutó completo con *Entorno de ejecución → Reiniciar y ejecutar todas* y las 10 gráficas quedaron guardadas en `visualizaciones/`.